# Tutorial 02: Anharmonic Potentials - DX Center Example

This tutorial demonstrates how to work with anharmonic potential energy surfaces,
using the DX center defect as an example.

**Learning objectives:**
- Load or generate anharmonic Q-E data
- Fit potentials using spline and Morse methods
- Assess fit quality quantitatively
- Compare harmonic vs anharmonic approximations
- Calculate carrier capture with anharmonic potentials

**Background:**
The DX center is a well-known deep-level defect in III-V semiconductors (e.g., AlGaAs).
It exhibits strong lattice relaxation with a highly anharmonic potential energy surface,
making it an excellent test case for anharmonic fitting methods.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

from carriercapture.core.potential import Potential
from carriercapture.core.config_coord import ConfigCoordinate
from carriercapture.visualization import (
    plot_potential,
    plot_capture_coefficient,
    plot_configuration_coordinate,
)

# Configure plots
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# For reproducibility
np.random.seed(42)

## 1. Generate Anharmonic Data

We'll create synthetic data representing a DX-center-like potential.
The DX center has a characteristic Morse-like shape with:
- Large configuration coordinate shift (ΔQ ~ 15-20 amu^0.5·Å)
- Asymmetric potential well
- Barrier for capture

In practice, this data would come from DFT calculations (e.g., via the doped package).

In [ ]:
# Generate synthetic DX center data for initial (excited) state
# Using a shifted Morse potential

def morse_potential(Q, D, a, Q0, E0=0):
    """Morse potential: V(Q) = D * (1 - exp(-a*(Q-Q0)))^2 + E0"""
    return D * (1 - np.exp(-a * (Q - Q0)))**2 + E0

# Initial state (excited) - shallow Morse well
Q_data_i = np.linspace(-5, 30, 40)
D_i = 1.5      # Dissociation energy (eV)
a_i = 0.25     # Width parameter (1/amu^0.5·Å)
Q0_i = 0.0     # Equilibrium position
E0_i = 0.8     # Energy offset (eV)

E_data_i = morse_potential(Q_data_i, D_i, a_i, Q0_i, E0_i)
# Add realistic noise (mimics DFT uncertainty)
E_data_i += np.random.normal(0, 0.015, len(Q_data_i))

# Final state (ground) - deeper Morse well, shifted
Q_data_f = np.linspace(-5, 30, 40)
D_f = 2.5      # Deeper well
a_f = 0.20     # Slightly wider
Q0_f = 15.0    # Large lattice relaxation
E0_f = 0.0     # Reference energy

E_data_f = morse_potential(Q_data_f, D_f, a_f, Q0_f, E0_f)
E_data_f += np.random.normal(0, 0.015, len(Q_data_f))

print(f"Generated data points:")
print(f"  Initial state: {len(Q_data_i)} points, Q range: [{Q_data_i.min():.1f}, {Q_data_i.max():.1f}]")
print(f"  Final state: {len(Q_data_f)} points, Q range: [{Q_data_f.min():.1f}, {Q_data_f.max():.1f}]")

In [ ]:
# Visualize raw data
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(Q_data_i, E_data_i, s=50, label='Initial state (DFT)', alpha=0.7, c='C0')
ax.scatter(Q_data_f, E_data_f, s=50, label='Final state (DFT)', alpha=0.7, c='C1')

ax.set_xlabel('Q (amu$^{0.5}$·Å)', fontsize=14)
ax.set_ylabel('E (eV)', fontsize=14)
ax.set_title('Raw DFT Data: DX Center Configuration Coordinate', fontsize=16)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Fit Potentials with Different Methods

We'll compare three fitting approaches:
1. **Spline**: Flexible interpolation, captures any shape
2. **Morse**: Physical model for anharmonic oscillators
3. **Harmonic**: Simple parabola (baseline comparison)

Each method has trade-offs in flexibility, physical interpretability, and robustness.

In [ ]:
# Create Potential objects for initial state
pot_i_spline = Potential(Q_data=Q_data_i, E_data=E_data_i, name="Initial (spline)")
pot_i_morse = Potential(Q_data=Q_data_i, E_data=E_data_i, name="Initial (Morse)")
pot_i_harmonic = Potential(Q_data=Q_data_i, E_data=E_data_i, name="Initial (harmonic)")

# Fit with different methods
pot_i_spline.fit(fit_type='spline', order=4, smoothness=0.01)
pot_i_morse.fit(fit_type='morse')
pot_i_harmonic.fit(fit_type='harmonic')

print("Initial state fit parameters:")
print(f"  Spline: order={pot_i_spline.fit_params.get('order', 'N/A')}")
print(f"  Morse: D={pot_i_morse.fit_params.get('D', 'N/A'):.3f} eV, "
      f"a={pot_i_morse.fit_params.get('a', 'N/A'):.3f}, "
      f"r0={pot_i_morse.fit_params.get('r0', 'N/A'):.3f}")
print(f"  Harmonic: k={pot_i_harmonic.fit_params.get('k', 'N/A'):.4f} eV/Å², "
      f"Q0={pot_i_harmonic.fit_params.get('Q0', 'N/A'):.3f}")

In [ ]:
# Same for final state
pot_f_spline = Potential(Q_data=Q_data_f, E_data=E_data_f, name="Final (spline)")
pot_f_morse = Potential(Q_data=Q_data_f, E_data=E_data_f, name="Final (Morse)")
pot_f_harmonic = Potential(Q_data=Q_data_f, E_data=E_data_f, name="Final (harmonic)")

pot_f_spline.fit(fit_type='spline', order=4, smoothness=0.01)
pot_f_morse.fit(fit_type='morse')
pot_f_harmonic.fit(fit_type='harmonic')

print("\nFinal state fit parameters:")
print(f"  Morse: D={pot_f_morse.fit_params.get('D', 'N/A'):.3f} eV, "
      f"a={pot_f_morse.fit_params.get('a', 'N/A'):.3f}, "
      f"r0={pot_f_morse.fit_params.get('r0', 'N/A'):.3f}")
print(f"  Harmonic: k={pot_f_harmonic.fit_params.get('k', 'N/A'):.4f} eV/Å², "
      f"Q0={pot_f_harmonic.fit_params.get('Q0', 'N/A'):.3f}")

## 3. Visualize and Compare Fits

In [ ]:
# Create fine grid for plotting fits
Q_fine = np.linspace(-5, 30, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Initial state
ax1 = axes[0]
ax1.scatter(Q_data_i, E_data_i, s=60, c='black', label='DFT data', zorder=5)
ax1.plot(Q_fine, pot_i_spline(Q_fine), '-', lw=2, label='Spline', alpha=0.8)
ax1.plot(Q_fine, pot_i_morse(Q_fine), '--', lw=2, label='Morse', alpha=0.8)
ax1.plot(Q_fine, pot_i_harmonic(Q_fine), ':', lw=2, label='Harmonic', alpha=0.8)
ax1.set_xlabel('Q (amu$^{0.5}$·Å)', fontsize=12)
ax1.set_ylabel('E (eV)', fontsize=12)
ax1.set_title('Initial State Fits', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(-0.2, 3.0)

# Final state
ax2 = axes[1]
ax2.scatter(Q_data_f, E_data_f, s=60, c='black', label='DFT data', zorder=5)
ax2.plot(Q_fine, pot_f_spline(Q_fine), '-', lw=2, label='Spline', alpha=0.8)
ax2.plot(Q_fine, pot_f_morse(Q_fine), '--', lw=2, label='Morse', alpha=0.8)
ax2.plot(Q_fine, pot_f_harmonic(Q_fine), ':', lw=2, label='Harmonic', alpha=0.8)
ax2.set_xlabel('Q (amu$^{0.5}$·Å)', fontsize=12)
ax2.set_ylabel('E (eV)', fontsize=12)
ax2.set_title('Final State Fits', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-0.2, 3.0)

plt.tight_layout()
plt.show()

## 4. Quantify Fit Quality

We'll calculate R² and root-mean-square error (RMSE) for each fit method.

In [ ]:
def calculate_fit_metrics(Q_data, E_data, potential):
    """Calculate R² and RMSE for a fit."""
    E_pred = potential(Q_data)
    
    # R² (coefficient of determination)
    ss_res = np.sum((E_data - E_pred)**2)
    ss_tot = np.sum((E_data - E_data.mean())**2)
    r2 = 1 - (ss_res / ss_tot)
    
    # RMSE
    rmse = np.sqrt(np.mean((E_data - E_pred)**2))
    
    # Max absolute error
    max_err = np.max(np.abs(E_data - E_pred))
    
    return {'R2': r2, 'RMSE': rmse, 'Max_Error': max_err}

# Calculate metrics for all fits
print("Fit Quality Metrics")
print("=" * 60)

print("\nInitial State:")
print(f"{'Method':<12} {'R²':>10} {'RMSE (meV)':>12} {'Max Err (meV)':>14}")
print("-" * 50)
for name, pot in [('Spline', pot_i_spline), ('Morse', pot_i_morse), ('Harmonic', pot_i_harmonic)]:
    m = calculate_fit_metrics(Q_data_i, E_data_i, pot)
    print(f"{name:<12} {m['R2']:>10.6f} {m['RMSE']*1000:>12.2f} {m['Max_Error']*1000:>14.2f}")

print("\nFinal State:")
print(f"{'Method':<12} {'R²':>10} {'RMSE (meV)':>12} {'Max Err (meV)':>14}")
print("-" * 50)
for name, pot in [('Spline', pot_f_spline), ('Morse', pot_f_morse), ('Harmonic', pot_f_harmonic)]:
    m = calculate_fit_metrics(Q_data_f, E_data_f, pot)
    print(f"{name:<12} {m['R2']:>10.6f} {m['RMSE']*1000:>12.2f} {m['Max_Error']*1000:>14.2f}")

## 5. Solve Schrödinger Equation

Now we solve for the phonon eigenstates using the spline fits (best quality).

In [ ]:
# Solve for eigenstates
pot_i_spline.solve(nev=150)
pot_f_spline.solve(nev=80)

print(f"Solved Schrödinger equation:")
print(f"  Initial state: {len(pot_i_spline.eigenvalues)} eigenvalues")
print(f"  Final state: {len(pot_f_spline.eigenvalues)} eigenvalues")

# Show first few eigenvalues
print(f"\nFirst 5 eigenvalues (initial state):")
for i in range(5):
    print(f"  n={i}: E = {pot_i_spline.eigenvalues[i]:.4f} eV")

print(f"\nFirst 5 eigenvalues (final state):")
for i in range(5):
    print(f"  n={i}: E = {pot_f_spline.eigenvalues[i]:.4f} eV")

In [ ]:
# Visualize potentials with wavefunctions
fig1 = plot_potential(pot_i_spline, show_wavefunctions=True, wf_sampling=5, title="Initial State (Spline Fit)")
fig1.show()

fig2 = plot_potential(pot_f_spline, show_wavefunctions=True, wf_sampling=5, title="Final State (Spline Fit)")
fig2.show()

## 6. Compare Anharmonic vs Harmonic Energy Levels

For a harmonic oscillator, energy levels are equally spaced: $E_n = \hbar\omega(n + 1/2)$.
For anharmonic potentials, the spacing changes with n.

In [ ]:
# Solve harmonic fits for comparison
pot_i_harmonic.solve(nev=150)
pot_f_harmonic.solve(nev=80)

# Calculate energy level spacings
def level_spacings(eigenvalues):
    return np.diff(eigenvalues)

spacings_spline = level_spacings(pot_i_spline.eigenvalues[:30])
spacings_harmonic = level_spacings(pot_i_harmonic.eigenvalues[:30])

fig, ax = plt.subplots(figsize=(10, 5))

n_values = np.arange(len(spacings_spline))
ax.plot(n_values, spacings_spline * 1000, 'o-', label='Anharmonic (spline)', markersize=6)
ax.plot(n_values, spacings_harmonic * 1000, 's--', label='Harmonic', markersize=6)

ax.set_xlabel('Quantum number n', fontsize=12)
ax.set_ylabel('Level spacing $E_{n+1} - E_n$ (meV)', fontsize=12)
ax.set_title('Anharmonicity: Energy Level Spacing vs n', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Harmonic level spacing: {spacings_harmonic.mean()*1000:.2f} ± {spacings_harmonic.std()*1000:.2f} meV (constant)")
print(f"Anharmonic spacing at n=0: {spacings_spline[0]*1000:.2f} meV")
print(f"Anharmonic spacing at n=20: {spacings_spline[20]*1000:.2f} meV")

## 7. Calculate Carrier Capture Coefficients

Now we calculate the capture coefficient using both anharmonic (spline) and harmonic potentials
to see the effect of anharmonicity on the final results.

In [ ]:
# Parameters
W = 0.05  # Electron-phonon coupling (eV) - typical for DX centers
volume = 1e-21  # Supercell volume (cm³)
temperature = np.linspace(100, 500, 50)

# Anharmonic calculation
cc_anharmonic = ConfigCoordinate(
    pot_i=pot_i_spline,
    pot_f=pot_f_spline,
    W=W,
    degeneracy=1,
)

# Find crossing point
Q0_crossing = 7.5  # Approximate crossing point

cc_anharmonic.calculate_overlap(Q0=Q0_crossing, cutoff=0.3, sigma=0.015)
cc_anharmonic.calculate_capture_coefficient(volume=volume, temperature=temperature)

print(f"Anharmonic calculation complete.")
print(f"  Overlap matrix shape: {cc_anharmonic.overlap_matrix.shape}")

In [ ]:
# Harmonic calculation for comparison
cc_harmonic = ConfigCoordinate(
    pot_i=pot_i_harmonic,
    pot_f=pot_f_harmonic,
    W=W,
    degeneracy=1,
)

cc_harmonic.calculate_overlap(Q0=Q0_crossing, cutoff=0.3, sigma=0.015)
cc_harmonic.calculate_capture_coefficient(volume=volume, temperature=temperature)

print(f"Harmonic calculation complete.")

In [ ]:
# Compare results
fig, ax = plt.subplots(figsize=(10, 6))

ax.semilogy(1000/temperature, cc_anharmonic.capture_coefficient, 'o-', 
            label='Anharmonic (spline)', markersize=5, linewidth=2)
ax.semilogy(1000/temperature, cc_harmonic.capture_coefficient, 's--', 
            label='Harmonic', markersize=5, linewidth=2, alpha=0.8)

ax.set_xlabel('1000/T (K$^{-1}$)', fontsize=14)
ax.set_ylabel('Capture Coefficient C (cm³/s)', fontsize=14)
ax.set_title('Effect of Anharmonicity on Carrier Capture', fontsize=16)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# Add temperature scale on top
ax2 = ax.twiny()
temp_ticks = [500, 400, 300, 200, 150]
ax2.set_xlim(ax.get_xlim())
ax2.set_xticks([1000/T for T in temp_ticks])
ax2.set_xticklabels([str(T) for T in temp_ticks])
ax2.set_xlabel('Temperature (K)', fontsize=12)

plt.tight_layout()
plt.show()

# Quantify difference
ratio = cc_anharmonic.capture_coefficient / cc_harmonic.capture_coefficient
print(f"\nCapture coefficient comparison at selected temperatures:")
print(f"{'T (K)':<8} {'C_anharm (cm³/s)':<18} {'C_harm (cm³/s)':<18} {'Ratio':>8}")
print("-" * 55)
for T in [150, 200, 300, 400, 500]:
    idx = np.argmin(np.abs(temperature - T))
    print(f"{T:<8} {cc_anharmonic.capture_coefficient[idx]:<18.3e} "
          f"{cc_harmonic.capture_coefficient[idx]:<18.3e} {ratio[idx]:>8.2f}")

## 8. Physical Interpretation

**Key observations:**

1. **Fit quality**: Spline and Morse fits capture the anharmonic shape well; harmonic fails at large Q.

2. **Energy level spacing**: 
   - Harmonic: constant spacing (ℏω)
   - Anharmonic: spacing decreases with n (levels bunch up near dissociation)

3. **Capture coefficient differences**:
   - Anharmonicity affects the density of states and Franck-Condon factors
   - The harmonic approximation may over- or under-estimate capture rates
   - Differences can be factors of 2-10× depending on the system

**When is the harmonic approximation sufficient?**
- Small lattice relaxation (ΔQ < 5 amu^0.5·Å)
- Shallow defects with small thermal barriers
- Qualitative trends and order-of-magnitude estimates

**When is anharmonic treatment necessary?**
- Deep defects like DX centers
- Large lattice relaxation (ΔQ > 10)
- Quantitative comparison with experiment
- Systems with asymmetric potentials

## Summary

In this tutorial, we:

- ✓ Generated synthetic anharmonic data (DX-center-like Morse potentials)
- ✓ Compared spline, Morse, and harmonic fitting methods
- ✓ Quantified fit quality with R² and RMSE metrics
- ✓ Solved the Schrödinger equation for anharmonic potentials
- ✓ Observed anharmonic effects on energy level spacing
- ✓ Calculated carrier capture coefficients
- ✓ Compared harmonic vs anharmonic results

**Next steps:**
- Try loading real DFT data using the `doped` integration
- Explore the interactive dashboard (Tutorial 04)
- Run parameter scans over anharmonicity parameters (Tutorial 03)

## Exercises

1. **Morse parameter sensitivity**: Vary the Morse parameters (D, a) and observe effects on capture
2. **Crossing point**: How does the choice of Q₀ affect the overlap calculation?
3. **Phonon frequency**: Extract effective ℏω from the harmonic fit and compare with literature
4. **Real data**: If you have DFT data, try loading it with `Potential.from_file()`

In [ ]:
# Exercise 1: Vary Morse parameter 'a' (anharmonicity strength)
a_values = [0.15, 0.20, 0.25, 0.30, 0.35]
results = []

for a_test in a_values:
    # Generate data with different anharmonicity
    Q_test = np.linspace(-5, 30, 40)
    E_test = morse_potential(Q_test, D=2.5, a=a_test, Q0=15.0, E0=0.0)
    E_test += np.random.normal(0, 0.015, len(Q_test))
    
    # Fit and solve
    pot_test = Potential(Q_data=Q_test, E_data=E_test)
    pot_test.fit(fit_type='spline', order=4, smoothness=0.01)
    pot_test.solve(nev=80)
    
    # Calculate capture (using fixed initial state)
    cc_test = ConfigCoordinate(pot_i=pot_i_spline, pot_f=pot_test, W=W)
    cc_test.calculate_overlap(Q0=Q0_crossing, cutoff=0.3, sigma=0.015)
    cc_test.calculate_capture_coefficient(volume=volume, temperature=np.array([300.0]))
    
    results.append(cc_test.capture_coefficient[0])

# Plot
plt.figure(figsize=(8, 5))
plt.semilogy(a_values, results, 'o-', markersize=10, linewidth=2)
plt.xlabel('Morse parameter a (1/amu$^{0.5}$·Å)', fontsize=12)
plt.ylabel('C(300K) (cm³/s)', fontsize=12)
plt.title('Effect of Anharmonicity Strength on Capture', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()